# 08 — RAG for Network Engineers

**Network LLM Engineering — Part II — Knowledge and Context**

### Learning goals
- Build a minimal RAG pipeline
- Separate retrieval quality from generation quality
- Understand chunking, grounding and citations

In [ ]:
%pip install -q transformers==5.14.1 datasets==5.0.1 accelerate==1.14.0 peft==0.20.0 trl==1.10.0 sentence-transformers==5.7.0 pandas matplotlib scikit-learn requests jsonschema

In [ ]:
from pathlib import Path

def find_root():
    for p in [Path.cwd(), Path.cwd().parent, Path("/content/network_llm_engineering_course")]:
        if (p / "data" / "glossary.csv").exists():
            return p
    raise FileNotFoundError("Run from the extracted network_llm_engineering_course folder.")

ROOT = find_root()
DATA = ROOT / "data"
print("Course root:", ROOT)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else (
    torch.float16 if torch.cuda.is_available() else torch.float32
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
).eval()

def render_chat(messages, generation=True):
    kw = dict(tokenize=False, add_generation_prompt=generation)
    try:
        return tokenizer.apply_chat_template(messages, enable_thinking=False, **kw)
    except TypeError:
        return tokenizer.apply_chat_template(messages, **kw)

@torch.inference_mode()
def generate(messages, max_new_tokens=160, temperature=0.0):
    text = render_chat(messages, True)
    toks = tokenizer(text, return_tensors="pt")
    dev = next(model.parameters()).device
    toks = {k:v.to(dev) for k,v in toks.items()}
    sample = temperature > 0
    out = model.generate(
        **toks, max_new_tokens=max_new_tokens,
        do_sample=sample,
        temperature=temperature if sample else None,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0, toks["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import json, numpy as np
from sentence_transformers import SentenceTransformer

docs = [json.loads(x) for x in open(DATA/"mini_network_knowledge.jsonl", encoding="utf-8")]
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
matrix = embedder.encode([d["text"] for d in docs], normalize_embeddings=True)

def retrieve(query, k=3):
    q = embedder.encode([query], normalize_embeddings=True)[0]
    scores = matrix @ q
    order = np.argsort(-scores)[:k]
    return [docs[i] | {"score":float(scores[i])} for i in order]

def rag_answer(question):
    hits = retrieve(question, 3)
    context = "\n\n".join(f"[{h['id']}] {h['text']}" for h in hits)
    messages = [
      {"role":"system","content":
       "Answer only from the supplied network context. If it is insufficient, say so. "
       "Cite source IDs in square brackets."},
      {"role":"user","content":f"CONTEXT:\n{context}\n\nQUESTION:\n{question}"}
    ]
    return hits, generate(messages, 180)

hits, answer = rag_answer("Why might a tunnel pass small packets but fail large ones?")
print([h["id"] for h in hits])
print(answer)

## Evaluate RAG in layers

A bad answer can come from:
1. **ingestion/chunking failure**
2. **retrieval failure**
3. **reranking failure**
4. **prompt/context construction failure**
5. **generation/faithfulness failure**

Do not respond to every RAG failure by fine-tuning the generator.

## Dynamic network facts

RAG/tooling is usually the right home for:
- inventory,
- running configs,
- current routing tables,
- maintenance windows,
- vendor advisories,
- internal runbooks,
- ticket history.

Weights are slow to update and hard to audit as a factual database.

### Exercise

Intentionally remove the MTU document and rerun the query.
Observe how a well-designed grounded assistant should say context is insufficient instead of confidently inventing an answer.